In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make src/ importable from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from src.dataset import load_combined_dataset
from src.train import train_per_horizon_models
from src.baselines import evaluate_naive_baselines
from src.features import build_supervised_dataset
from src.train import temporal_split

In [8]:
# 1. Load
prices, exog = load_combined_dataset(
    storage_account="sagreekdamdevweu",
    join="inner",
)
print(f"Training on {len(prices)} hours, {prices.index.min()} → {prices.index.max()}")
print(f"Exog columns: {list(exog.columns)}")
print()

Training on 29328 hours, 2022-12-31 22:00:00+00:00 → 2026-05-06 21:00:00+00:00
Exog columns: ['load_forecast_mw', 'solar', 'wind_onshore']



In [9]:
prices.tail(60)

2026-05-04 10:00:00+00:00     -0.0100
2026-05-04 11:00:00+00:00     -0.0550
2026-05-04 12:00:00+00:00     -0.0050
2026-05-04 13:00:00+00:00      0.0025
2026-05-04 14:00:00+00:00     52.4525
2026-05-04 15:00:00+00:00    134.0475
2026-05-04 16:00:00+00:00    171.9950
2026-05-04 17:00:00+00:00    220.7650
2026-05-04 18:00:00+00:00    255.5875
2026-05-04 19:00:00+00:00    219.3550
2026-05-04 20:00:00+00:00    163.2200
2026-05-04 21:00:00+00:00    170.1400
2026-05-04 22:00:00+00:00    162.4250
2026-05-04 23:00:00+00:00    144.5000
2026-05-05 00:00:00+00:00    125.0700
2026-05-05 01:00:00+00:00    125.2125
2026-05-05 02:00:00+00:00    126.9875
2026-05-05 03:00:00+00:00    139.4075
2026-05-05 04:00:00+00:00    150.5200
2026-05-05 05:00:00+00:00    121.6000
2026-05-05 06:00:00+00:00     23.7400
2026-05-05 07:00:00+00:00     -0.0050
2026-05-05 08:00:00+00:00      0.0000
2026-05-05 09:00:00+00:00      0.0000
2026-05-05 10:00:00+00:00      0.0000
2026-05-05 11:00:00+00:00      0.0000
2026-05-05 1

In [11]:
import plotly.express as px

fig = px.line(
    x=prices.index,
    y=prices.values,
    labels={"x": "Date", "y": "Price"},
    title="Prices over time",
)
fig.show()

In [10]:
# 2. Train (this is where features.py is internally invoked)
result = train_per_horizon_models(
    prices,
    exog=exog,
    gate_closure_hour=12,
    horizons=tuple(range(0, 24)),
    same_hour_lag_days=(1, 2, 7),
    context_window=2,
    test_days=30,
)
print(result.summary())
print()

TrainResult Summary
  Models trained:    24 per-horizon LightGBM models
  Features:          44
  Gate closure:      12:00
  Overall test MAE:  17.15 EUR/MWh
  Overall test RMSE: 22.21 EUR/MWh
  Best horizon:      h=11 (MAE 11.46)
  Worst horizon:     h=14 (MAE 28.06)



In [11]:
# 3. Naive baselines for comparison
X, y, meta = build_supervised_dataset(
    prices, exog=exog,
    gate_closure_hour=12,
    horizons=tuple(range(0, 24)),
    same_hour_lag_days=(1, 2, 7),
    context_window=2,
)
X

,ft_price_minus_0h,ft_price_minus_1h,ft_price_minus_2h,ft_price_minus_3h,ft_mean_24h,ft_std_24h,ft_max_24h,ft_min_24h,ft_mean_168h,ft_std_168h,...,target_is_holiday,target_hour_sin,target_hour_cos,target_dow_sin,target_dow_cos,target_load_forecast_mw,target_solar,target_wind_onshore,target_net_load_mw,horizon
0,81.9500,84.9000,168.5000,177.2500,200.246000,54.846832,268.1900,81.9500,200.246000,54.846832,...,0,0.000000,1.000000,0.000000,1.000000,3562.0,0.0,527.0,3035.0,0
1,188.8900,187.9100,188.8900,196.2300,250.290417,68.863390,363.1500,179.7300,231.042564,67.726661,...,0,0.000000,1.000000,0.781831,0.623490,4107.0,0.0,151.0,3956.0,0
2,174.7600,175.0000,180.2000,191.5800,279.277083,71.240242,363.7700,172.7800,249.417619,72.467718,...,0,0.000000,1.000000,0.974928,-0.222521,4199.0,0.0,240.0,3959.0,0
3,172.0000,169.4100,173.5000,183.8300,231.874583,75.783884,369.1700,130.0000,244.578161,73.376974,...,0,0.000000,1.000000,0.433884,-0.900969,4267.0,0.0,883.0,3384.0,0
4,165.1000,165.0000,184.3800,192.9400,224.231250,82.200167,362.3100,100.0000,240.178829,75.452277,...,1,0.000000,1.000000,-0.433884,-0.900969,4284.0,0.0,575.0,3709.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29273,1.2850,0.4875,1.0000,4.0650,94.993542,63.261921,194.1175,0.0025,74.440193,66.819939,...,1,-0.258819,0.965926,-0.433884,-0.900969,4042.5,0.0,3152.5,890.0,23
29274,0.0000,0.0000,0.0000,0.0000,71.499479,60.979326,171.7150,0.0000,73.622515,65.517441,...,0,-0.258819,0.965926,-0.974928,-0.222521,4040.0,0.0,2432.5,1607.5,23
29275,-4.4675,-3.9150,-3.9025,-2.9725,51.541458,46.790272,105.8475,-4.4675,71.635818,61.934064,...,0,-0.258819,0.965926,-0.781831,0.623490,3940.0,0.0,1872.5,2067.5,23
29276,-8.0950,-9.9925,-5.2625,-0.9575,65.937188,57.966634,132.3325,-9.9925,73.723943,59.765894,...,0,-0.258819,0.965926,0.000000,1.000000,4070.0,0.0,487.5,3582.5,23


In [24]:
X.columns

Index(['ft_price_minus_0h', 'ft_price_minus_1h', 'ft_price_minus_2h',
       'ft_price_minus_3h', 'ft_mean_24h', 'ft_std_24h', 'ft_max_24h',
       'ft_min_24h', 'ft_mean_168h', 'ft_std_168h', 'ft_max_168h',
       'ft_min_168h', 'ft_load_forecast_mw', 'ft_solar', 'ft_wind_onshore',
       'ft_net_load_mw', 'tr_lag_d1_k-2', 'tr_lag_d1_k-1', 'tr_lag_d1_k+0',
       'tr_lag_d1_k+1', 'tr_lag_d1_k+2', 'tr_lag_d2_k-2', 'tr_lag_d2_k-1',
       'tr_lag_d2_k+0', 'tr_lag_d2_k+1', 'tr_lag_d2_k+2', 'tr_lag_d7_k-2',
       'tr_lag_d7_k-1', 'tr_lag_d7_k+0', 'tr_lag_d7_k+1', 'tr_lag_d7_k+2',
       'target_hour', 'target_dow', 'target_month', 'target_is_weekend',
       'target_is_holiday', 'target_hour_sin', 'target_hour_cos',
       'target_dow_sin', 'target_dow_cos', 'target_load_forecast_mw',
       'target_solar', 'target_wind_onshore', 'target_net_load_mw', 'horizon'],
      dtype='str')

In [23]:
prices.tail(48)

2026-05-04 22:00:00+00:00    162.4250
2026-05-04 23:00:00+00:00    144.5000
2026-05-05 00:00:00+00:00    125.0700
2026-05-05 01:00:00+00:00    125.2125
2026-05-05 02:00:00+00:00    126.9875
2026-05-05 03:00:00+00:00    139.4075
2026-05-05 04:00:00+00:00    150.5200
2026-05-05 05:00:00+00:00    121.6000
2026-05-05 06:00:00+00:00     23.7400
2026-05-05 07:00:00+00:00     -0.0050
2026-05-05 08:00:00+00:00      0.0000
2026-05-05 09:00:00+00:00      0.0000
2026-05-05 10:00:00+00:00      0.0000
2026-05-05 11:00:00+00:00      0.0000
2026-05-05 12:00:00+00:00      0.0000
2026-05-05 13:00:00+00:00      0.0000
2026-05-05 14:00:00+00:00     54.8125
2026-05-05 15:00:00+00:00    134.9925
2026-05-05 16:00:00+00:00    159.2125
2026-05-05 17:00:00+00:00    211.9675
2026-05-05 18:00:00+00:00    238.9475
2026-05-05 19:00:00+00:00    157.1175
2026-05-05 20:00:00+00:00    141.6175
2026-05-05 21:00:00+00:00    151.4700
2026-05-05 22:00:00+00:00    164.9400
2026-05-05 23:00:00+00:00    149.3550
2026-05-06 0

In [27]:
meta.head(50)

,forecast_time,horizon,target_time
0,2023-01-01 12:00:00+00:00,0,2023-01-02 00:00:00+00:00
1,2023-01-02 12:00:00+00:00,0,2023-01-03 00:00:00+00:00
2,2023-01-03 12:00:00+00:00,0,2023-01-04 00:00:00+00:00
3,2023-01-04 12:00:00+00:00,0,2023-01-05 00:00:00+00:00
4,2023-01-05 12:00:00+00:00,0,2023-01-06 00:00:00+00:00
5,2023-01-06 12:00:00+00:00,0,2023-01-07 00:00:00+00:00
6,2023-01-07 12:00:00+00:00,0,2023-01-08 00:00:00+00:00
7,2023-01-08 12:00:00+00:00,0,2023-01-09 00:00:00+00:00
8,2023-01-09 12:00:00+00:00,0,2023-01-10 00:00:00+00:00
9,2023-01-10 12:00:00+00:00,0,2023-01-11 00:00:00+00:00


In [33]:
prices[meta['forecast_time'].iloc[-5]]

np.float64(1.285)

In [12]:
_, _, _, _, y_te, m_te = temporal_split(X, y, meta, test_days=30)
print(f"Feature matrix shape: {X.shape}")
print(f"Feature columns: {list(X.columns)}")
print()

Feature matrix shape: (29278, 45)
Feature columns: ['ft_price_minus_0h', 'ft_price_minus_1h', 'ft_price_minus_2h', 'ft_price_minus_3h', 'ft_mean_24h', 'ft_std_24h', 'ft_max_24h', 'ft_min_24h', 'ft_mean_168h', 'ft_std_168h', 'ft_max_168h', 'ft_min_168h', 'ft_load_forecast_mw', 'ft_solar', 'ft_wind_onshore', 'ft_net_load_mw', 'tr_lag_d1_k-2', 'tr_lag_d1_k-1', 'tr_lag_d1_k+0', 'tr_lag_d1_k+1', 'tr_lag_d1_k+2', 'tr_lag_d2_k-2', 'tr_lag_d2_k-1', 'tr_lag_d2_k+0', 'tr_lag_d2_k+1', 'tr_lag_d2_k+2', 'tr_lag_d7_k-2', 'tr_lag_d7_k-1', 'tr_lag_d7_k+0', 'tr_lag_d7_k+1', 'tr_lag_d7_k+2', 'target_hour', 'target_dow', 'target_month', 'target_is_weekend', 'target_is_holiday', 'target_hour_sin', 'target_hour_cos', 'target_dow_sin', 'target_dow_cos', 'target_load_forecast_mw', 'target_solar', 'target_wind_onshore', 'target_net_load_mw', 'horizon']



In [13]:
m_te

,forecast_time,horizon,target_time
0,2026-04-06 12:00:00+00:00,0,2026-04-07 00:00:00+00:00
1,2026-04-07 12:00:00+00:00,0,2026-04-08 00:00:00+00:00
2,2026-04-08 12:00:00+00:00,0,2026-04-09 00:00:00+00:00
3,2026-04-09 12:00:00+00:00,0,2026-04-10 00:00:00+00:00
4,2026-04-10 12:00:00+00:00,0,2026-04-11 00:00:00+00:00
...,...,...,...
713,2026-04-30 12:00:00+00:00,23,2026-05-01 23:00:00+00:00
714,2026-05-01 12:00:00+00:00,23,2026-05-02 23:00:00+00:00
715,2026-05-02 12:00:00+00:00,23,2026-05-03 23:00:00+00:00
716,2026-05-03 12:00:00+00:00,23,2026-05-04 23:00:00+00:00


In [14]:
naive = evaluate_naive_baselines(prices, m_te, y_te, gate_closure_hour=12)
weights = naive["n_test"]
naive_avg = (naive["mae_average"] * weights).sum() / weights.sum()
naive_yest = (naive["mae_yesterday"] * weights).sum() / weights.sum()
naive_lwk = (naive["mae_last_week"] * weights).sum() / weights.sum()

In [17]:
weights

0     30
1     30
2     30
3     30
4     30
5     30
6     30
7     30
8     30
9     30
10    30
11    30
12    30
13    30
14    30
15    30
16    30
17    30
18    30
19    30
20    30
21    30
22    29
23    29
Name: n_test, dtype: int64

In [15]:
print(f"Comparison:")
print(f"  Naïve yesterday-same-hour: {naive_yest:6.2f} EUR/MWh")
print(f"  Naïve last-week-same-hour: {naive_lwk:6.2f} EUR/MWh")
print(f"  Naïve average:             {naive_avg:6.2f} EUR/MWh")
print(f"  LightGBM:                  {result.overall_test_mae:6.2f} EUR/MWh")

Comparison:
  Naïve yesterday-same-hour:  22.19 EUR/MWh
  Naïve last-week-same-hour:  27.09 EUR/MWh
  Naïve average:              21.11 EUR/MWh
  LightGBM:                   17.15 EUR/MWh


In [16]:
improvement = (naive_avg - result.overall_test_mae) / naive_avg * 100
print(f"\n  LGBM beats naïve average by: {improvement:.1f}%")


  LGBM beats naïve average by: 18.7%


In [18]:
naive

,horizon,mae_yesterday,mae_last_week,mae_average,rmse_yesterday,rmse_last_week,rmse_average,n_test
0,0,9.480333,13.427250,10.052958,13.225976,21.172002,13.732670,30
1,1,11.417333,14.752000,11.705417,15.460732,21.983577,15.362749,30
2,2,11.025500,16.130167,12.040333,15.124596,24.822800,16.786235,30
3,3,10.281833,12.717333,10.275667,13.501260,19.342863,14.097278,30
4,4,20.670083,15.938750,15.134667,26.496954,22.181181,20.104018,30
5,5,30.373583,32.246000,26.213458,40.391007,39.575324,32.554546,30
6,6,37.014250,48.420667,33.778792,50.960614,65.520638,43.885497,30
7,7,23.689000,37.534917,26.019542,38.596595,62.242117,39.975892,30
8,8,14.418833,32.317083,21.156625,30.172150,55.606918,33.759511,30
9,9,20.816250,38.974917,28.857583,44.595126,61.498513,40.920642,30


In [34]:
# Per-horizon breakdown
print("\nPer-horizon performance:")
print(result.metrics_per_horizon.to_string(index=False))

# Feature importance from one of the trained models (h=12, midday)
import pandas as pd

model = result.models[12]  # noon target
importance = pd.DataFrame({
    "feature": result.feature_names,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print("\nTop 15 features (h=12 model):")
print(importance.head(15).to_string(index=False))

# Same for h=20 (evening peak)
model = result.models[20]
importance = pd.DataFrame({
    "feature": result.feature_names,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print("\nTop 15 features (h=20 model):")
print(importance.head(15).to_string(index=False))


Per-horizon performance:
 horizon       mae      rmse  n_test
       0 14.155811 16.946680      30
       1 16.607216 19.322395      30
       2 12.380553 16.576737      30
       3 11.561642 14.684510      30
       4 14.271791 18.887540      30
       5 16.985650 21.242834      30
       6 21.606324 28.318445      30
       7 18.516863 23.237550      30
       8 13.696664 19.132827      30
       9 13.887558 23.403203      30
      10 12.167491 21.556856      30
      11 11.459549 19.078106      30
      12 17.863443 22.334572      30
      13 20.616704 28.112265      30
      14 28.058635 37.966236      30
      15 23.985876 28.281476      30
      16 16.987225 21.341486      30
      17 23.907272 28.195550      30
      18 24.722031 30.300452      30
      19 16.312816 20.478427      30
      20 13.338047 16.174455      30
      21 16.902481 20.254554      30
      22 16.300576 19.651247      29
      23 15.193028 17.299260      29

Top 15 features (h=12 model):
                fe